# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment. Colab ships with DuckDB, so this is usually instant.


In [1]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder.** Run locally, the notebook starts inside `work/notebooks/`, so it steps up until it reaches the top folder, where the `.env` file lives. On Colab it works in Colab's own folder.
2. **Loads the tools.** `duckdb` runs SQL directly on the warehouse files over the internet, `pandas` and `numpy` handle the small results, and `scipy` supplies the two probability models used for the luck test (`poisson` and `nbinom`, the negative binomial).
3. **Reads the Hugging Face token** from, in order: an environment variable, the project's `.env` file, a Colab Secret named `HF_TOKEN`, and only as a last resort a typed prompt. DuckDB then logs in with `PROVIDER credential_chain`, which reads the token from the environment itself, so the token never appears in a query, an output, or the notebook file.
4. **Defines the months.** Three development pairs are listed explicitly. June 2026 is never in any list, so no cell can read it.
5. **Defines `cached()`**, which runs a heavy warehouse query once, saves the small result as a Parquet file under `work/outputs/w03_cache/` (git ignores `*.parquet`), and reuses it on every later run. It also defines `show()` and `show_table()`, which display every result as a titled table.


In [2]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import nbinom, poisson

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

**Connected to the warehouse**

| Measure | Value |
| --- | --- |
| token read from | Colab Secret |
| token shown anywhere | no (DuckDB reads it from the environment) |
| development pairs | 2026-02 → 2026-03, 2026-03 → 2026-04, 2026-04 → 2026-05 |
| sealed pair, never read | 2026-05 → 2026-06 |
| cache folder | work/outputs/w03_cache |

## 1. Unit of analysis + time window

**One row = one page in one month pair.** The page's search data in month M (used by the features and the score) is paired with its search data in month M+1 (used only by the target). The warehouse stores page-days; the notebook adds them up to page-months inside DuckDB, then pairs consecutive months.

| Pair   | Month M: features and score | Month M+1: target | Role                                   |
| ------ | --------------------------- | ----------------- | -------------------------------------- |
| 1      | February 2026               | March 2026        | Build the rules                        |
| 2      | March 2026                  | April 2026        | Check that the rules hold              |
| 3      | April 2026                  | May 2026          | Check that the rules hold              |
| Sealed | May 2026                    | June 2026         | Final test, never read while designing |

- **Why calendar months:** the daily table is stored as one folder per month, so each pair reads exactly two folders, and a month is the review cycle an editor works in.
- **Why these months:** the history is far from balanced. Most clients only arrive in the last eight months (the coverage table below shows it), so earlier months have too few clients to learn from.
- **Why June stays sealed:** June is the last month in the data, so it is the natural outcome month of any "what happens next" question. Designing rules while looking at it would mean designing inside the test.

**Output, in one sentence:** each month, the analysis hands an editor the 50 pages most likely to keep falling well short of the clicks their position should earn, with the clicks at stake and a reason code.

**Method:** this cell counts the rows of every warehouse table and compares them with the documented release. Row counts come from the Parquet file footers, so it reads no data values.


In [3]:
# Documented sizes of build v20260703; a mismatch means the notebook is pointed at different data.
DOCUMENTED = {"dim_clients": 104, "dim_content": 519_606, "fact_content_daily_performance": 78_835_655,
              "fact_content_query_90d": 2_414_248}
SOURCES = {"dim_clients": DIM_CLIENTS, "dim_content": DIM_CONTENT, "fact_content_daily_performance": DAILY_FOOTERS,
           "fact_content_query_90d": QUERY_90D}
counts = {name: con.sql(f"SELECT COUNT(*) FROM {source}").fetchone()[0] for name, source in SOURCES.items()}
for name, rows in counts.items():
    assert rows == DOCUMENTED[name], f"{name}: {rows:,} rows, documented {DOCUMENTED[name]:,}"

show_table("Warehouse tables, counted against the documented release", pd.DataFrame({
    "Rows counted": [f"{counts[name]:,}" for name in DOCUMENTED],
    "Rows documented": [f"{rows:,}" for rows in DOCUMENTED.values()],
    "Match": ["yes"] * len(DOCUMENTED),
}, index=pd.Index(list(DOCUMENTED), name="Table")))

**Warehouse tables, counted against the documented release**

| Table | Rows counted | Rows documented | Match |
| --- | ---: | ---: | --- |
| dim_clients | 104 | 104 | yes |
| dim_content | 519,606 | 519,606 | yes |
| fact_content_daily_performance | 78,835,655 | 78,835,655 | yes |
| fact_content_query_90d | 2,414,248 | 2,414,248 | yes |

**Finding:** all four tables match the documented release exactly, including the 78,835,655 daily rows. The notebook is reading build v20260703, the same data the documentation describes.


**Method:** this cell scans the daily table month by month, from January 2025 to May 2026 (June is left out on purpose), and counts how many clients and pages had at least one impression in each month, plus the first and last day with impressions. It runs once and is cached.


In [4]:
# Clients and pages with impressions per month; the sums are kept to reconcile later tables against.
coverage = cached("coverage_by_month", f"""
    SELECT month,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages,
           SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
    FROM {month_files(PANEL_MONTHS)}
    WHERE gsc_impressions > 0
    GROUP BY month
    ORDER BY month""")
show_table("Clients and pages with search impressions, by month", pd.DataFrame({
    "First day": coverage["first_day"].dt.strftime("%Y-%m-%d").to_numpy(),
    "Last day": coverage["last_day"].dt.strftime("%Y-%m-%d").to_numpy(),
    "Clients": coverage["clients"].to_numpy(),
    "Pages": coverage["pages"].map("{:,}".format).to_numpy(),
}, index=pd.Index(coverage["month"], name="Month")))

**Clients and pages with search impressions, by month**

| Month | First day | Last day | Clients | Pages |
| --- | --- | --- | ---: | ---: |
| 2025-01 | 2025-01-27 | 2025-01-31 | 2 | 476 |
| 2025-02 | 2025-02-01 | 2025-02-28 | 3 | 5,903 |
| 2025-03 | 2025-03-01 | 2025-03-31 | 4 | 10,374 |
| 2025-04 | 2025-04-01 | 2025-04-30 | 4 | 13,046 |
| 2025-05 | 2025-05-01 | 2025-05-31 | 4 | 14,887 |
| 2025-06 | 2025-06-01 | 2025-06-30 | 9 | 16,399 |
| 2025-07 | 2025-07-01 | 2025-07-31 | 16 | 27,945 |
| 2025-08 | 2025-08-01 | 2025-08-31 | 15 | 37,204 |
| 2025-09 | 2025-09-01 | 2025-09-30 | 23 | 53,127 |
| 2025-10 | 2025-10-01 | 2025-10-31 | 28 | 64,906 |
| 2025-11 | 2025-11-01 | 2025-11-30 | 39 | 99,403 |
| 2025-12 | 2025-12-01 | 2025-12-31 | 38 | 109,639 |
| 2026-01 | 2026-01-01 | 2026-01-31 | 37 | 121,544 |
| 2026-02 | 2026-02-01 | 2026-02-28 | 46 | 153,559 |
| 2026-03 | 2026-03-01 | 2026-03-31 | 47 | 176,738 |
| 2026-04 | 2026-04-01 | 2026-04-30 | 51 | 194,760 |
| 2026-05 | 2026-05-01 | 2026-05-31 | 56 | 237,910 |

**Finding:** the history is heavily back-loaded. The first five months (January to May 2025) hold only 2 to 4 clients each, and the data starts on 2025-01-27, as documented. From November 2025 onward, 37 or more clients are present every month, and the development months hold 46 to 56 clients and 153,559 to 237,910 pages. Every month listed has data from its first calendar day to its last, apart from January 2025, where the release begins.


## 2. Fields: feature / label / context / excluded

A data contract has to say what every field is for before any model sees it. Two levels matter:

- **Raw columns.** Every column of the three tables this project reads gets exactly one use, with a reason. The code below holds that list and refuses to run if any column is missing, listed twice, or new.
- **Row fields.** The fields built for each page-pair row go into exactly one of four buckets. **Feature:** known by the end of month M, safe to learn from. **Label:** the target or anything it is computed from. **Context:** for joining, grouping, splitting, and reading results, never learned from (IDs live here). **Excluded:** future information, product decisions, or fields that fail a check, each with a reason.

The same raw column can feed two buckets through different months. Clicks in month M are a feature; clicks in month M+1 are a label input. That is why row fields carry a `_m` or `_next` suffix, and why one check below rejects any feature ending in `_next`.

**Method:** this cell writes the raw-column contract as a table, then checks it against the actual column lists of the daily table (one month's folder is enough, since every month shares the same columns), the content table, and the client table.


In [5]:
# Each entry: table, the columns it covers, how they are used, and why.
RAW_CONTRACT = [
    ("daily", ["report_date", "month"], "Window building", "Places each day in its month"),
    ("daily", ["client_hash_id", "content_hash_id"], "Context", "Pseudonymous IDs: joining, grouping, client-grouped splits"),
    ("daily", ["gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_avg_position"], "Feature source (M) and label source (M+1)",
     "Search Console measurements; month M builds features, month M+1 builds the target"),
    ("daily", ["gsc_data_available"], "Not used", "Only says whether the page had impressions that day (section 3c)"),
    ("daily", ["client_has_gsc"], "Not used", "Client eligibility uses dim_clients.gsc_data_start instead"),
    ("daily", ["client_has_ga4", "ga4_data_available"], "Context", "Mark when GA4 numbers are real; three-valued, filtered with IS TRUE"),
    ("daily", ["ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions", "ga4_total_engagement_sec",
               "scroll_events"], "Context", "Engagement reason code only; never ranks pages"),
    ("daily", ["sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid"], "Excluded",
     "Other traffic channels, outside a search click-through question"),
    ("daily", ["sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other"],
     "Excluded", "AI-referral traffic: a different lane, and sparse"),
    ("content", ["client_hash_id", "content_hash_id", "keyword_hash_id", "url_hash_id"], "Context", "Pseudonymous IDs"),
    ("content", ["content_type", "main_intent", "search_volume", "competition", "cpc", "keyword_token_count"], "Feature",
     "Keyword research fields, fixed before the page is written; missing values get a has-value flag"),
    ("content", ["keyword_char_count", "competition_level"], "Excluded", "Duplicates of keyword_token_count and competition"),
    ("content", ["content_created_date", "keyword_created_date"], "Context", "Page and keyword age, for reading results"),
    ("content", ["content_updated_date", "word_count", "char_count", "backlinks", "category_count", "url_char_count"],
     "Excluded", "Snapshot values from July 2026 that may describe the page after later edits (section 3g)"),
    ("content", ["is_published", "is_deleted"], "Excluded", "The page's state in July 2026: future information"),
    ("content", ["provider_used", "model_used", "optimization_eligible_date"], "Excluded", "FlyRank product decisions"),
    ("content", ["last_optimized_date"], "Filter only", "Pages FlyRank optimized during a pair are removed from it"),
    ("clients", ["client_hash_id"], "Context", "Pseudonymous ID"),
    ("clients", ["gsc_data_start"], "Filter only", "A client's search history must cover all of month M"),
    ("clients", ["ga4_data_start", "has_gsc_access", "has_ga4_access", "access_profile"], "Context", "Coverage checks"),
    ("clients", ["is_active", "client_updated_date"], "Excluded", "Status at the July 2026 snapshot: future information"),
    ("clients", ["client_created_date"], "Not used", "Account creation says nothing about clicks"),
]
ACTUAL = {
    "daily": month_files(["2026-03"]),
    "content": DIM_CONTENT,
    "clients": DIM_CLIENTS,
}
for table, source in ACTUAL.items():
    listed = [column for entry in RAW_CONTRACT if entry[0] == table for column in entry[1]]
    actual = set(con.sql(f"DESCRIBE SELECT * FROM {source}").df()["column_name"])
    assert len(listed) == len(set(listed)), f"{table}: a column is listed twice."
    assert set(listed) == actual, f"{table}: unlisted {sorted(actual - set(listed))}, unknown {sorted(set(listed) - actual)}"

TABLE_NAME = {"daily": "fact_content_daily_performance", "content": "dim_content", "clients": "dim_clients"}
show_table("Raw-column contract (every column of the three tables, each listed once)", pd.DataFrame({
    "Table": [TABLE_NAME[entry[0]] for entry in RAW_CONTRACT],
    "Columns": [", ".join(entry[1]) for entry in RAW_CONTRACT],
    "Use": [entry[2] for entry in RAW_CONTRACT],
    "Why": [entry[3] for entry in RAW_CONTRACT],
}))

**Raw-column contract (every column of the three tables, each listed once)**

| Table | Columns | Use | Why |
| --- | --- | --- | --- |
| fact_content_daily_performance | report_date, month | Window building | Places each day in its month |
| fact_content_daily_performance | client_hash_id, content_hash_id | Context | Pseudonymous IDs: joining, grouping, client-grouped splits |
| fact_content_daily_performance | gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position | Feature source (M) and label source (M+1) | Search Console measurements; month M builds features, month M+1 builds the target |
| fact_content_daily_performance | gsc_data_available | Not used | Only says whether the page had impressions that day (section 3c) |
| fact_content_daily_performance | client_has_gsc | Not used | Client eligibility uses dim_clients.gsc_data_start instead |
| fact_content_daily_performance | client_has_ga4, ga4_data_available | Context | Mark when GA4 numbers are real; three-valued, filtered with IS TRUE |
| fact_content_daily_performance | ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events | Context | Engagement reason code only; never ranks pages |
| fact_content_daily_performance | sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid | Excluded | Other traffic channels, outside a search click-through question |
| fact_content_daily_performance | sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other | Excluded | AI-referral traffic: a different lane, and sparse |
| dim_content | client_hash_id, content_hash_id, keyword_hash_id, url_hash_id | Context | Pseudonymous IDs |
| dim_content | content_type, main_intent, search_volume, competition, cpc, keyword_token_count | Feature | Keyword research fields, fixed before the page is written; missing values get a has-value flag |
| dim_content | keyword_char_count, competition_level | Excluded | Duplicates of keyword_token_count and competition |
| dim_content | content_created_date, keyword_created_date | Context | Page and keyword age, for reading results |
| dim_content | content_updated_date, word_count, char_count, backlinks, category_count, url_char_count | Excluded | Snapshot values from July 2026 that may describe the page after later edits (section 3g) |
| dim_content | is_published, is_deleted | Excluded | The page's state in July 2026: future information |
| dim_content | provider_used, model_used, optimization_eligible_date | Excluded | FlyRank product decisions |
| dim_content | last_optimized_date | Filter only | Pages FlyRank optimized during a pair are removed from it |
| dim_clients | client_hash_id | Context | Pseudonymous ID |
| dim_clients | gsc_data_start | Filter only | A client's search history must cover all of month M |
| dim_clients | ga4_data_start, has_gsc_access, has_ga4_access, access_profile | Context | Coverage checks |
| dim_clients | is_active, client_updated_date | Excluded | Status at the July 2026 snapshot: future information |
| dim_clients | client_created_date | Not used | Account creation says nothing about clicks |

**Finding:** every one of the 66 columns in the three tables (31 in the daily table, 26 in the content table, 9 in the client table) has exactly one use, and the checks confirm that no column is missing, listed twice, or unknown. Only ten columns can ever reach a model: four Search Console measurements in month M and six keyword-research fields.


The fourth table, `fact_content_query_90d`, is **excluded as a source of features**, every column of it. It covers one fixed window, checked in section 3h. For the development pairs that window lies after month M and overlaps the target months, and no training pair has an equivalent. It stays available for one job: reading the top picks afterwards, for example to see whether a flagged page's impressions come from one dominant query.

**Method:** this cell writes the row-field buckets and checks the rules that keep them honest: no field in two buckets, no feature built from month M+1, and the planted leak from section 6 never among the features.


In [6]:
# Row fields of one page-pair row; the suffix _m means month M and _next means month M+1.
FIVE_FEATURES = ["impressions_m", "clicks_m", "position_m", "position_volatility_m", "content_type"]
BUCKETS = {
    "Feature": FIVE_FEATURES + ["main_intent", "search_volume", "competition", "cpc", "keyword_token_count"],
    "Label": ["impressions_next", "clicks_next", "position_next", "expected_next", "p_next", "target"],
    "Context": ["client_hash_id", "content_hash_id", "month_m", "ga4_sessions_m"],
    "Excluded": ["position_two_months"],
}
everything = [field for fields in BUCKETS.values() for field in fields]
assert len(everything) == len(set(everything)), "A field sits in two buckets."
assert not [field for field in BUCKETS["Feature"] if field.endswith("_next")], "A feature is built from month M+1."
assert "position_two_months" not in BUCKETS["Feature"], "The planted leak is among the features."

WHY = {
    "Feature": "Known by the last day of month M",
    "Label": "Measured in month M+1; never an input",
    "Context": "Joining, grouping, splitting, reason codes; never learned from",
    "Excluded": "Position averaged over M and M+1: the overlap leak measured in section 6",
}
show_table("Row-field buckets (each field in exactly one)", pd.DataFrame({
    "Fields": [", ".join(fields) for fields in BUCKETS.values()],
    "Rule": [WHY[bucket] for bucket in BUCKETS],
}, index=pd.Index(list(BUCKETS), name="Bucket")))

**Row-field buckets (each field in exactly one)**

| Bucket | Fields | Rule |
| --- | --- | --- |
| Feature | impressions_m, clicks_m, position_m, position_volatility_m, content_type, main_intent, search_volume, competition, cpc, keyword_token_count | Known by the last day of month M |
| Label | impressions_next, clicks_next, position_next, expected_next, p_next, target | Measured in month M+1; never an input |
| Context | client_hash_id, content_hash_id, month_m, ga4_sessions_m | Joining, grouping, splitting, reason codes; never learned from |
| Excluded | position_two_months | Position averaged over M and M+1: the overlap leak measured in section 6 |

**Finding:** the row has 10 feature fields, 6 label fields, 4 context fields, and 1 excluded field, with no field in two buckets and no feature built from month M+1. The five-feature frame is the first five feature fields; the five keyword fields join it when a model is trained.


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query here. The heavy part comes first: one scan per development month, adding page-days up to one row per page per month. The checks after it read that small table or a single month of daily rows.

### 3a. The page-month table

**Method:** this cell scans each of the four development months once and keeps, per page: impressions, clicks, the pieces needed for an impression-weighted position and its spread (summed only over days with a valid position, 1 or more, since Google positions start at 1), the clicks earned on days without a valid position, the number of days with impressions, and GA4 sessions counted only where `ga4_data_available IS TRUE`. Each month is cached as its own file. It then checks the grain and reconciles the totals against the coverage table.


In [7]:
# One row per page per month; position pieces only from days with a valid position (1 or more).
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)

assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."
totals = page_months.groupby("month")[["impressions", "clicks"]].sum()
reference = coverage.set_index("month").loc[DEV_MONTHS, ["impressions", "clicks"]]
assert (totals == reference).all().all(), "Page-month totals do not reconcile with the coverage scan."

show_table("Page-month table: grain and reconciliation", pd.DataFrame({
    "Page rows": page_months.groupby("month").size().map("{:,}".format),
    "Impressions": totals["impressions"].map("{:,.0f}".format),
    "Matches coverage scan": "yes",
}).rename_axis("Month"))

**Page-month table: grain and reconciliation**

| Month | Page rows | Impressions | Matches coverage scan |
| --- | ---: | ---: | --- |
| 2026-02 | 153,559 | 180,128,922 | yes |
| 2026-03 | 176,738 | 280,657,589 | yes |
| 2026-04 | 194,760 | 292,067,218 | yes |
| 2026-05 | 237,910 | 266,164,899 | yes |

**Finding:** the four development months add up to 153,559, 176,738, 194,760, and 237,910 page rows, one per page per month, and their impressions match the coverage scan to the last impression in every month. The aggregation lost and duplicated nothing.


### 3b. The daily grain

**Method:** this cell checks, on March 2026, that no page appears twice on the same date (the documented grain is date × client × page), and that no page belongs to more than one client. It also checks the content table's grain: one row per page.


In [8]:
# Zero rows back from each probe means the grain holds.
march = month_files(["2026-03"])
daily_dupes = cached("grain_daily_march", f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n FROM {march}
    GROUP BY ALL HAVING COUNT(*) > 1 LIMIT 5""")
two_clients = cached("grain_page_client_march", f"""
    SELECT content_hash_id FROM {march}
    GROUP BY content_hash_id HAVING COUNT(DISTINCT client_hash_id) > 1 LIMIT 5""")
content_rows, content_pages = con.sql(f"SELECT COUNT(*), COUNT(DISTINCT content_hash_id) FROM {DIM_CONTENT}").fetchone()
assert daily_dupes.empty and two_clients.empty and content_rows == content_pages, "A grain check failed."

show("Grain checks (all passed)", {
    "pages appearing twice on one date (March)": len(daily_dupes),
    "pages belonging to two clients (March)": len(two_clients),
    "content table: rows vs distinct pages": f"{content_rows:,} vs {content_pages:,}",
})

**Grain checks (all passed)**

| Measure | Value |
| --- | --- |
| pages appearing twice on one date (March) | 0 |
| pages belonging to two clients (March) | 0 |
| content table: rows vs distinct pages | 519,606 vs 519,606 |

**Finding:** all three grain checks pass. No page appears twice on the same date, no page belongs to two clients, and the content table holds exactly one row for each of its 519,606 pages.


### 3c. What the availability flags really mean

**Method:** this cell counts March's page-days by the two availability flags and, for each combination, how many have zero impressions or missing values. The documentation calls the flags three-valued (TRUE, FALSE, or empty).


In [9]:
# Every flag combination with its row count and how its numbers look.
flags = cached("flag_states_march", f"""
    SELECT gsc_data_available, ga4_data_available, COUNT(*) AS days,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
           SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS empty_sessions
    FROM {march}
    GROUP BY ALL ORDER BY ALL""")
gsc_true = flags["gsc_data_available"].eq(True)
assert (flags.loc[gsc_true, "zero_impressions"] == 0).all(), "A TRUE search flag has a zero-impression day."
assert (flags.loc[~gsc_true, "zero_impressions"] == flags.loc[~gsc_true, "days"]).all(), "A FALSE search flag has impressions."

show_table("March 2026 page-days by availability flag", pd.DataFrame({
    "Search flag": flags["gsc_data_available"].map({True: "TRUE", False: "FALSE"}).to_numpy(),
    "GA4 flag": flags["ga4_data_available"].map({True: "TRUE", False: "FALSE"}).fillna("empty").to_numpy(),
    "Page-days": flags["days"].map("{:,}".format).to_numpy(),
    "With zero impressions": flags["zero_impressions"].map("{:,.0f}".format).to_numpy(),
    "With empty GA4 sessions": flags["empty_sessions"].map("{:,.0f}".format).to_numpy(),
}))

**March 2026 page-days by availability flag**

| Search flag | GA4 flag | Page-days | With zero impressions | With empty GA4 sessions |
| --- | --- | ---: | ---: | ---: |
| FALSE | FALSE | 4,690,323 | 4,690,323 | 0 |
| FALSE | TRUE | 49,619 | 49,619 | 0 |
| FALSE | empty | 1,490,375 | 1,490,375 | 1,490,375 |
| TRUE | FALSE | 1,718,348 | 0 | 0 |
| TRUE | TRUE | 364,347 | 0 | 0 |
| TRUE | empty | 1,528,366 | 0 | 1,528,366 |

**Finding:** the search flag `gsc_data_available` does not mean "search data is connected". It is TRUE on exactly the 3,611,061 page-days that had impressions and FALSE on all 6,230,317 page-days that had none, so it adds nothing beyond `gsc_impressions > 0`, and treating FALSE as "missing" would wrongly discard every zero-impression day. The GA4 flag really is three-valued: 413,966 page-days are TRUE (4.2% of March), and 3,018,741 are empty, with empty session counts. Filtering with `IS TRUE` is the only safe reading.


### 3d. Position traps

**Method:** this cell checks two things about position. First, whether `gsc_sum_position` is exactly the daily position × impressions (if so, summing it gives an exact impression-weighted position). Second, how many page-days have impressions but an impossible position: exactly 0, or between 0 and 1 (Google positions start at 1, so both mean the position was not recorded properly). Both kinds are left out of the position average, and the cell measures how much of the traffic sits on those days across the four development months.


In [10]:
# Is sum_position exactly position x impressions, and how many days have impressions but an impossible position?
position_check = cached("position_checks_march_v2", f"""
    SELECT COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ABS(gsc_sum_position - gsc_avg_position * gsc_impressions) < 1e-6 THEN 1 ELSE 0 END) AS identity_holds,
           SUM(CASE WHEN gsc_avg_position <= 0 OR gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS position_zero_days,
           SUM(CASE WHEN gsc_avg_position > 0 AND gsc_avg_position < 1 THEN 1 ELSE 0 END) AS position_below_one_days,
           SUM(CASE WHEN gsc_avg_position > 0 AND gsc_avg_position < 1 THEN gsc_impressions ELSE 0 END) AS below_one_impressions,
           SUM(gsc_impressions) AS impressions
    FROM {march}
    WHERE gsc_impressions > 0""").iloc[0]
assert position_check["identity_holds"] == position_check["days_with_impressions"], "sum_position is not position x impressions."

share_impressions = 1 - page_months["position_impressions"].sum() / page_months["impressions"].sum()
share_clicks = page_months["clicks_without_position"].sum() / page_months["clicks"].sum()
show("Position checks", {
    "March days where sum_position = position x impressions": f"{position_check['identity_holds']:,.0f} of {position_check['days_with_impressions']:,.0f}",
    "March days with impressions but position 0": f"{position_check['position_zero_days']:,.0f}",
    "March days with a position between 0 and 1": f"{position_check['position_below_one_days']:,.0f}",
    "share of March impressions on days between 0 and 1": f"{position_check['below_one_impressions'] / position_check['impressions']:.2%}",
    "share of impressions on days without a valid position (Feb to May)": f"{share_impressions:.2%}",
    "share of clicks on days without a valid position (Feb to May)": f"{share_clicks:.2%}",
})

**Position checks**

| Measure | Value |
| --- | --- |
| March days where sum_position = position x impressions | 3,611,061 of 3,611,061 |
| March days with impressions but position 0 | 163,189 |
| March days with a position between 0 and 1 | 101,548 |
| share of March impressions on days between 0 and 1 | 2.36% |
| share of impressions on days without a valid position (Feb to May) | 1.94% |
| share of clicks on days without a valid position (Feb to May) | 1.07% |

**Finding:** `gsc_sum_position` equals daily position × impressions on all 3,611,061 March page-days with impressions, so summing it gives an exact impression-weighted position. Two kinds of day carry an impossible position: 163,189 March page-days show 0, and 101,548 show a value between 0 and 1 (Google positions start at 1). The second kind is the dangerous one: it sits mostly on busy days, holding 2.36% of March's impressions, and it made pages look as if they ranked at the very top, which inflated their expected clicks. Both kinds are now left out of the position average. Across February to May, days without a valid position hold 1.94% of impressions and 1.07% of clicks.


### 3e. Windows and client history

**Method:** this cell checks that each development month has data on its first and last calendar day, then counts, for each pair, the clients with impressions in month M and how many of them have search history (`gsc_data_start`) covering all of month M.


In [11]:
# Each month's first and last day with impressions, from the coverage scan.
months = coverage.set_index("month").loc[DEV_MONTHS]
for month, row in months.iterrows():
    start = pd.Timestamp(f"{month}-01")
    assert row["first_day"] == start and row["last_day"] == start + pd.offsets.MonthEnd(0), f"{month} is not complete."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
rows = {}
for month_m, month_next in PAIRS:
    active = page_months.loc[page_months["month"] == month_m, "client_hash_id"].unique()
    covered = clients.loc[clients["client_hash_id"].isin(active) & (clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"))]
    rows[f"{month_m} → {month_next}"] = {"Clients with impressions in M": len(active), "History covers all of M": len(covered)}

show_table("Every development month is complete; clients whose history covers month M", pd.DataFrame(rows).T.rename_axis("Pair"))

**Every development month is complete; clients whose history covers month M**

| Pair | Clients with impressions in M | History covers all of M |
| --- | ---: | ---: |
| 2026-02 → 2026-03 | 46 | 35 |
| 2026-03 → 2026-04 | 47 | 42 |
| 2026-04 → 2026-05 | 51 | 46 |

**Finding:** all four development months run from their first to their last calendar day. Requiring a client's history to cover all of month M keeps 35 of the 46 clients active in February, 42 of 47 in March, and 46 of 51 in April. The February pair loses the most, because the search history of 11 of its clients begins after 1 February.


### 3f. Missing values and their pattern

**Method:** this cell measures, for the keyword fields planned as features, how often each is empty, overall and separately for each content type. If the gaps follow content type, filling them with 0 would quietly teach a model the content type, so they get a has-value flag instead.


In [12]:
# Share of empty values per keyword field, overall and by content type.
KEYWORD_FIELDS = ["search_volume", "competition", "cpc", "main_intent", "keyword_token_count"]
empty = cached("keyword_missing_by_type", f"""
    SELECT content_type, COUNT(*) AS pages,
           {", ".join(f"AVG(CASE WHEN {field} IS NULL THEN 1.0 ELSE 0 END) AS {field}" for field in KEYWORD_FIELDS)}
    FROM {DIM_CONTENT}
    GROUP BY content_type ORDER BY pages DESC""").set_index("content_type")
overall = empty[KEYWORD_FIELDS].mul(empty["pages"], axis=0).sum() / empty["pages"].sum()
table = pd.concat([overall.to_frame("All pages").T, empty[KEYWORD_FIELDS]])
show_table("Share of pages with an empty keyword field, by content type",
           table.map("{:.0%}".format).assign(Pages=[f"{empty['pages'].sum():,}"] + empty["pages"].map("{:,}".format).tolist())
           .rename_axis("Content type"))

**Share of pages with an empty keyword field, by content type**

| Content type | search_volume | competition | cpc | main_intent | keyword_token_count | Pages |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| All pages | 27% | 27% | 27% | 29% | 0% | 519,606 |
| keyword article | 19% | 19% | 19% | 20% | 0% | 459,174 |
| feedly article | 100% | 100% | 100% | 100% | 0% | 57,024 |
| comparison article | 0% | 0% | 0% | 0% | 0% | 3,408 |

**Finding:** the gaps follow content type exactly. Feedly articles have no keyword data at all (100% empty), keyword articles are 19 to 20% empty, and comparison articles are complete. `keyword_token_count` is never empty. Filling the gaps with 0 would hand a model a hidden copy of the content type, so each keyword field gets a has-value flag instead.


**Method:** this cell measures how much of the development data has real GA4 engagement numbers, counting a page-month as covered only when it has GA4 sessions recorded on days where `ga4_data_available IS TRUE`.


In [13]:
# GA4 coverage of page-months, which bounds how often an engagement reason code can apply.
ga4 = page_months.assign(has_ga4=page_months["ga4_sessions"] > 0, has_30=page_months["ga4_sessions"] >= 30)
show_table("Page-months with real GA4 sessions", pd.DataFrame({
    "Page-months": ga4.groupby("month").size().map("{:,}".format),
    "Any GA4 sessions": ga4.groupby("month")["has_ga4"].mean().map("{:.1%}".format),
    "30 or more sessions": ga4.groupby("month")["has_30"].mean().map("{:.1%}".format),
}).rename_axis("Month"))

**Page-months with real GA4 sessions**

| Month | Page-months | Any GA4 sessions | 30 or more sessions |
| --- | ---: | ---: | ---: |
| 2026-02 | 153,559 | 14.4% | 1.7% |
| 2026-03 | 176,738 | 36.0% | 5.5% |
| 2026-04 | 194,760 | 41.2% | 3.7% |
| 2026-05 | 237,910 | 40.0% | 6.8% |

**Finding:** real GA4 sessions exist on 14.4% of February's page-months and 36.0 to 41.2% of later months, but only 1.7 to 6.8% of page-months reach 30 sessions, the starter pipeline's minimum for an engagement reason code. Engagement can explain why a page is flagged for only a small minority of pages, which is why it stays a reason code and never ranks pages.


### 3g. When the content table was captured

**Method:** this cell checks how many pages were edited after the first development month began, and in which months FlyRank's recorded optimizations fall. Both show whether content-table values can be trusted to describe a page as it was in month M.


In [14]:
# How much of the content table changed after February 2026, and when optimizations happened.
snapshot = con.sql(f"""
    SELECT AVG(CASE WHEN content_updated_date >= DATE '2026-02-01' THEN 1.0 ELSE 0 END) AS edited_after,
           MAX(content_updated_date) AS last_edit,
           AVG(CASE WHEN last_optimized_date IS NOT NULL THEN 1.0 ELSE 0 END) AS has_optimization
    FROM {DIM_CONTENT}""").df().iloc[0]
optimized = con.sql(f"""
    SELECT strftime(last_optimized_date, '%Y-%m') AS month, COUNT(*) AS pages
    FROM {DIM_CONTENT} WHERE last_optimized_date IS NOT NULL GROUP BY ALL ORDER BY ALL""").df()

show("Content-table timing", {
    "pages edited on or after 2026-02-01": f"{snapshot['edited_after']:.1%}",
    "latest edit date in the table": f"{snapshot['last_edit']:%Y-%m-%d}",
    "pages with a recorded optimization": f"{snapshot['has_optimization']:.1%}",
})
show_table("Pages by month of their latest recorded optimization", optimized.assign(
    pages=optimized["pages"].map("{:,}".format)).set_index("month").rename_axis("Month").rename(columns={"pages": "Pages"}))

**Content-table timing**

| Measure | Value |
| --- | --- |
| pages edited on or after 2026-02-01 | 80.4% |
| latest edit date in the table | 2026-07-06 |
| pages with a recorded optimization | 8.7% |

**Pages by month of their latest recorded optimization**

| Month | Pages |
| --- | ---: |
| 2026-04 | 612 |
| 2026-05 | 17,099 |
| 2026-06 | 22,721 |
| 2026-07 | 4,964 |

**Finding:** 80.4% of pages were edited on or after 2026-02-01, and the latest edit is 2026-07-06, after every development month. Word counts and similar fields in this table may therefore describe a page after it changed, which is why they are excluded. FlyRank's recorded optimizations (8.7% of pages) all fall between April and July 2026, mostly in May and June.


### 3h. The query table's window

**Method:** this cell reads the query table's window dates.


In [15]:
# The query table covers one fixed window for every row.
window = con.sql(f"""
    SELECT MIN(window_start) AS first_start, MAX(window_start) AS last_start,
           MIN(window_end) AS first_end, MAX(window_end) AS last_end FROM {QUERY_90D}""").df().iloc[0]
assert window["first_start"] == window["last_start"] and window["first_end"] == window["last_end"], "The window is not fixed."
show("Query table window", {
    "window start": f"{window['first_start']:%Y-%m-%d}",
    "window end": f"{window['first_end']:%Y-%m-%d}",
    "same window on every row": "yes",
})

**Query table window**

| Measure | Value |
| --- | --- |
| window start | 2026-04-02 |
| window end | 2026-06-30 |
| same window on every row | yes |

**Finding:** every row of the query table covers 2026-04-02 to 2026-06-30. For the February and March pairs that window lies entirely after month M, and for April to May it overlaps the target month, so excluding it from features is required, not optional.


## 4. The month-pair table and the five features

### 4a. Building the rows

**Method:** this cell defines how a month pair becomes rows, then builds the three development pairs. In order, a page is kept when it has impressions in both months, its client's history covers all of month M, it has a valid position in both months (month M+1 needs one to place it in a position bin), and FlyRank did not optimize it during the pair. It then computes:

- **the five features** from month M;
- **expected clicks in month M+1** from the pooled click rate of the page's position bin, scaled by its client factor (the target's expectation);
- **the baseline score's expectation in month M**: the five tiers from the earlier notebooks, also client-scaled;
- **the planted leak**: the same tier expectation, but with position averaged over both months.


In [16]:
# Position bins: rounded positions 1 to 10 one by one, then 11-20, 21-50, and beyond 50.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the baseline: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor.

    The client factor is the client's actual clicks divided by its expected clicks, with +1 on both
    sides so a client with very few clicks cannot receive an extreme factor.
    """
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


content = con.sql(f"SELECT content_hash_id, content_type, last_optimized_date FROM {DIM_CONTENT}").df()


def build_pair(month_m: str, month_next: str) -> tuple[pd.DataFrame, dict[str, int]]:
    """Return one month pair's rows and the number of rows left after each eligibility step."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    funnel = {"Pages with impressions in M": len(first), "... and in M+1": len(rows)}

    covered = clients.loc[clients["gsc_data_start"] <= start, "client_hash_id"]
    rows = rows[rows["client_hash_id"].isin(covered)]
    funnel["... client history covers all of M"] = len(rows)
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    funnel["... valid position in both months"] = len(rows)
    rows = rows.merge(content, on="content_hash_id", how="left")
    optimized = pd.to_datetime(rows["last_optimized_date"]).between(start, end)
    rows = rows[~optimized].copy()
    funnel["... not optimized during the pair"] = len(rows)

    for suffix in ("m", "next"):
        rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows["sum_position_sq_m"] / rows["position_impressions_m"] - rows["position_m"] ** 2
    rows["position_volatility_m"] = np.sqrt(spread.clip(lower=0))
    rows["position_two_months"] = ((rows["sum_position_m"] + rows["sum_position_next"])
                                   / (rows["position_impressions_m"] + rows["position_impressions_next"]))
    rows["expected_next"] = client_aware_expected(rows, "clicks_next", "impressions_next", position_bin(rows["position_next"]))
    rows["expected_m_tier"] = client_aware_expected(rows, "clicks_m", "impressions_m", position_tier(rows["position_m"]))
    rows["expected_m_leak"] = client_aware_expected(rows, "clicks_m", "impressions_m", position_tier(rows["position_two_months"]))
    rows["month_m"] = month_m
    return rows, funnel


built = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
pairs = {name: rows for name, (rows, _) in built.items()}
for rows in pairs.values():
    assert not rows.duplicated("content_hash_id").any(), "A page appears twice in one pair."

show_table("Rows left after each eligibility step", pd.DataFrame(
    {name: {step: f"{count:,}" for step, count in funnel.items()} for name, (_, funnel) in built.items()}).rename_axis("Step"))

**Rows left after each eligibility step**

| Step | 2026-02 → 2026-03 | 2026-03 → 2026-04 | 2026-04 → 2026-05 |
| --- | ---: | ---: | ---: |
| Pages with impressions in M | 153,559 | 176,738 | 194,760 |
| ... and in M+1 | 134,238 | 158,549 | 183,345 |
| ... client history covers all of M | 116,366 | 157,629 | 180,643 |
| ... valid position in both months | 114,972 | 156,165 | 173,369 |
| ... not optimized during the pair | 114,972 | 155,707 | 157,530 |

**Finding:** the first pair keeps 114,972 rows, the second 155,707, and the third 157,530. Each step has a visible cost. Requiring impressions in both months removes 19,321 February pages. The client-history rule removes 17,872 more in February but under 3,000 in later months. Requiring a valid position (1 or more) in both months removes 1,394, 1,464, and 7,274 rows. The optimization rule removes nothing in February, 458 rows in March, and 15,839 rows in April, matching the optimization dates above: they only begin in April 2026.


### 4b. The five features

**Method:** this cell summarizes the five features on the first development pair: how many rows have a value, and the median and 90th percentile of each number, then how the rows split by content type.


In [17]:
# The five features on February 2026, the month the first pair's score is allowed to see.
feb = pairs["2026-02 → 2026-03"]
numbers = feb[FIVE_FEATURES[:-1]]
show_table("Five-feature frame, February 2026: the four numbers", pd.DataFrame({
    "Rows with a value": numbers.notna().sum().map("{:,}".format),
    "Median": numbers.median().map("{:,.1f}".format),
    "90th percentile": numbers.quantile(0.9).map("{:,.1f}".format),
}).rename_axis("Feature"))
types = feb["content_type"].fillna("empty").value_counts()
show_table("Five-feature frame, February 2026: content type", pd.DataFrame({
    "Rows": types.map("{:,}".format), "Share": (types / types.sum()).map("{:.1%}".format)}).rename_axis("Content type"))

**Five-feature frame, February 2026: the four numbers**

| Feature | Rows with a value | Median | 90th percentile |
| --- | ---: | ---: | ---: |
| impressions_m | 114,972 | 210.0 | 3,612.9 |
| clicks_m | 114,972 | 0.0 | 10.0 |
| position_m | 114,972 | 8.2 | 32.7 |
| position_volatility_m | 114,972 | 3.4 | 16.9 |

**Five-feature frame, February 2026: content type**

| Content type | Rows | Share |
| --- | ---: | ---: |
| keyword article | 106,011 | 92.2% |
| feedly article | 8,457 | 7.4% |
| comparison article | 504 | 0.4% |

**Finding:** in February, half the pages earn no click at all (median clicks 0, median impressions 210), so a page-level click-through rate is mostly noise, which is why there is no impression cutoff and a luck test decides instead. The median page sits at position 8.2, and its daily position moves by a typical 3.4 places within the month, up to 16.9 for the most volatile tenth. That spread was invisible in the starter file's single 90-day average. Keyword articles make up 92.2% of rows, feedly articles 7.4%, and comparison articles 0.4%.


## 5. Is the luck test too confident?

The target counts a page as a real shortfall when its clicks in month M+1 fall far enough below expected that luck rarely explains it. The earlier notebooks measured "rarely" with the Poisson model, which assumes a page's clicks vary only by pure chance. Daily data makes that assumption testable.

**The test.** Take pages whose position barely moved between the two months (less than one place). If such a page's underlying click rate stayed the same, pure chance decides how its clicks split between the two months, and the spread of that split is known exactly. The **dispersion ratio** compares the actual spread with that known spread, after correcting for the overall change in click rate between the months. Poisson is right if the ratio is about 1.

**Method:** this cell computes the dispersion ratio on the first development pair, separately for pages with more and fewer clicks.


In [18]:
def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share


stable, share = stable_pages(feb)
total = stable["clicks_m"] + stable["clicks_next"]
ratio = (stable["clicks_next"] - total * share) ** 2 / (total * share * (1 - share))
band = pd.cut(total, [10, 30, 100, 300, 1000, np.inf], right=False,
              labels=["10 to 29", "30 to 99", "100 to 299", "300 to 999", "1,000 or more"])
show_table("Dispersion ratio by clicks across both months (Poisson expects 1.00)", pd.DataFrame({
    "Pages": band.value_counts(sort=False).map("{:,}".format),
    "Dispersion ratio": ratio.groupby(band, observed=True).mean().map("{:.2f}".format),
}).rename_axis("Clicks in both months"))

**Dispersion ratio by clicks across both months (Poisson expects 1.00)**

| Clicks in both months | Pages | Dispersion ratio |
| --- | ---: | ---: |
| 10 to 29 | 5,394 | 1.48 |
| 30 to 99 | 3,763 | 2.01 |
| 100 to 299 | 1,217 | 3.85 |
| 300 to 999 | 282 | 8.61 |
| 1,000 or more | 26 | 41.12 |

**Finding:** Poisson is too confident, and more so as pages get bigger. Pages with 10 to 29 clicks vary 1.48 times as much as pure chance allows; pages with 1,000 or more vary 41.12 times as much. This is the pattern of a negative binomial, where extra noise grows with the square of a page's clicks, and it matters because big pages hold most of the missed clicks.


**Method:** this cell estimates the negative binomial's spread factor α on each development pair, using the same stable pages. It compares the actual squared gap between each page's month-M+1 clicks and what its month-M rate predicts with the gap Poisson alone would produce, and attributes the excess to noise that grows with the square of the page's clicks. It then builds the target both ways, Poisson and negative binomial with α fixed at the median of the three estimates rounded to three decimals, and compares how often each calls a shortfall real.


In [19]:
def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)


def shortfall_p(clicks: pd.Series, expected: pd.Series, alpha: float) -> np.ndarray:
    """Return P(clicks this low or lower) under a negative binomial with mean `expected` and spread factor alpha."""
    return nbinom.cdf(clicks, 1 / alpha, 1 / (1 + alpha * expected))


alphas = {name: spread_factor(rows) for name, rows in pairs.items()}
# Locked as one exact, citable number: the median estimate rounded to three decimals.
ALPHA = round(float(np.median(list(alphas.values()))), 3)
summary = {}
for name, rows in pairs.items():
    below_half = rows["clicks_next"] <= 0.5 * rows["expected_next"]
    poisson_target = (poisson.cdf(rows["clicks_next"], rows["expected_next"]) < 0.05) & below_half
    rows["p_next"] = shortfall_p(rows["clicks_next"], rows["expected_next"], ALPHA)
    rows["target"] = (rows["p_next"] < 0.05) & below_half
    rows["missed_next"] = (rows["expected_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
    summary[name] = {
        "Spread factor α": f"{alphas[name]:.3f}",
        "Base rate, Poisson": f"{poisson_target.mean():.1%}",
        "Base rate, negative binomial": f"{rows['target'].mean():.1%}",
        "Positives, negative binomial": f"{rows['target'].sum():,}",
    }

show_table(f"Target base rate under each luck test (locked α = {ALPHA:.3f})", pd.DataFrame(summary).T.rename_axis("Pair"))

**Target base rate under each luck test (locked α = 0.039)**

| Pair | Spread factor α | Base rate, Poisson | Base rate, negative binomial | Positives, negative binomial |
| --- | ---: | ---: | ---: | ---: |
| 2026-02 → 2026-03 | 0.050 | 8.2% | 7.4% | 8,465 |
| 2026-03 → 2026-04 | 0.032 | 7.1% | 6.4% | 10,031 |
| 2026-04 → 2026-05 | 0.039 | 4.6% | 4.1% | 6,525 |

**Finding:** the spread factor is consistent across pairs (0.050, 0.032, and 0.039), and the median, 0.039, is locked. Switching the luck test from Poisson to the negative binomial withdraws about one positive in ten, mostly big pages whose shortfall natural variation can explain: the base rate falls from 8.2% to 7.4%, from 7.1% to 6.4%, and from 4.6% to 4.1%. The target stays usable, with 6,525 to 10,031 positives per pair. **Locked target:** clicks in month M+1 at most half of the client-aware, position-bin expectation, with negative binomial p below 0.05 at α = 0.039.


## 6. The leakage test

The starter file's position was a 90-day average that overlapped both 30-day windows, and the earlier framing named that overlap as its main limitation without being able to measure it. Here it can be measured. The baseline score from the earlier notebooks (missed clicks in month M, five tiers, client-scaled, kept only when Poisson p < 0.01) is run three ways and judged against the locked target:

- **Honest:** position from month M only.
- **Overlap leak:** position averaged over months M and M+1, which is what the starter's 90-day column did.
- **Blunt leak:** ranking by month M+1's own missed clicks, the extreme case, shown for contrast.

**Method:** this cell computes Precision@50, Precision@20, and Capture@50 for all three rankings on every development pair, with the base rate beside them.


In [20]:
def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the share of the top-k rows by score whose target is true."""
    return float(target.loc[score.sort_values(ascending=False).index[:k]].mean())


def capture_at_k(score: pd.Series, value: pd.Series, k: int) -> float:
    """Return the share of the total value (here, missed clicks on real shortfalls) held by the top-k rows."""
    return float(value.loc[score.sort_values(ascending=False).index[:k]].sum() / value.sum())


def baseline_score(rows: pd.DataFrame, expected: str) -> pd.Series:
    """Return month-M missed clicks, kept only where Poisson says luck explains the gap less than 1% of the time."""
    missed = (rows[expected] - rows["clicks_m"]).clip(lower=0)
    return missed.where(poisson.cdf(rows["clicks_m"], rows[expected]) < 0.01, 0)


results = []
for name, rows in pairs.items():
    rankings = {
        "Honest (month M position)": baseline_score(rows, "expected_m_tier"),
        "Overlap leak (position over M and M+1)": baseline_score(rows, "expected_m_leak"),
        "Blunt leak (month M+1 missed clicks)": (rows["expected_next"] - rows["clicks_next"]).clip(lower=0),
    }
    for ranking, score in rankings.items():
        results.append({"Pair": name, "Ranking": ranking,
                        "Precision@50": f"{precision_at_k(score, rows['target'], 50):.2f}",
                        "Precision@20": f"{precision_at_k(score, rows['target'], 20):.2f}",
                        "Capture@50": f"{capture_at_k(score, rows['missed_next'], 50):.1%}",
                        "Base rate": f"{rows['target'].mean():.1%}"})
show_table("Baseline score with and without a planted leak", pd.DataFrame(results).set_index(["Pair", "Ranking"]))

**Baseline score with and without a planted leak**

| Pair | Ranking | Precision@50 | Precision@20 | Capture@50 | Base rate |
| --- | --- | ---: | ---: | ---: | ---: |
| 2026-02 → 2026-03 | Honest (month M position) | 0.64 | 0.70 | 3.9% | 7.4% |
| 2026-02 → 2026-03 | Overlap leak (position over M and M+1) | 0.68 | 0.65 | 4.1% | 7.4% |
| 2026-02 → 2026-03 | Blunt leak (month M+1 missed clicks) | 0.84 | 0.75 | 7.2% | 7.4% |
| 2026-03 → 2026-04 | Honest (month M position) | 0.76 | 0.85 | 4.6% | 6.4% |
| 2026-03 → 2026-04 | Overlap leak (position over M and M+1) | 0.72 | 0.80 | 4.5% | 6.4% |
| 2026-03 → 2026-04 | Blunt leak (month M+1 missed clicks) | 0.86 | 0.90 | 8.0% | 6.4% |
| 2026-04 → 2026-05 | Honest (month M position) | 0.84 | 0.85 | 6.8% | 4.1% |
| 2026-04 → 2026-05 | Overlap leak (position over M and M+1) | 0.82 | 0.85 | 6.8% | 4.1% |
| 2026-04 → 2026-05 | Blunt leak (month M+1 missed clicks) | 0.92 | 0.95 | 12.1% | 4.1% |

**Finding:** the overlap leak is small and has no consistent direction: averaging position over both months moves Precision@50 by 0.04 or less (0.64 to 0.68, 0.76 to 0.72, 0.84 to 0.82). The starter file's 90-day position, named as the main limitation of the earlier framing, did not systematically inflate its results. The blunt leak shows what real leakage looks like: ranking by month M+1's own missed clicks lifts Precision@50 to 0.84 to 0.92 and nearly doubles Capture@50. The honest baseline is already strong, at 0.64, 0.76, and 0.84 against base rates of 7.4%, 6.4%, and 4.1%, roughly 9 to 20 times better than picking pages at random. That is the bar a learned model has to clear.


**Method:** this cell removes the planted leak column from every pair and re-checks that nothing built from month M+1 is left among the features.


In [21]:
# Delete the leak and confirm the features are back to month-M information only.
for rows in pairs.values():
    rows.drop(columns=["position_two_months", "expected_m_leak"], inplace=True)
    assert set(FIVE_FEATURES) <= set(rows.columns), "A feature column is missing."
    assert "position_two_months" not in rows.columns, "The leak column is still present."
assert not [field for field in FIVE_FEATURES if field.endswith("_next")], "A feature is built from month M+1."
show("Leak removed", {
    "leak column left in any pair": "no",
    "features built from month M+1": "none",
    "number carried forward": "the honest baseline row in the table above",
})

**Leak removed**

| Measure | Value |
| --- | --- |
| leak column left in any pair | no |
| features built from month M+1 | none |
| number carried forward | the honest baseline row in the table above |

**Finding:** the leak column is gone from every pair, and the five features hold month-M information only. The honest baseline is the number carried forward.


## 7. Limitations

What this data can never tell me, and the limits of the choices above:

- **Clicks, not causes.** The data records what happened. A page that keeps falling short is a candidate for review, not proof that a rewrite will recover clicks. Google's AI answers, rich results, brand searches, and a mismatch with what searchers want all lower click-through, and none of them is visible here.
- **An unbalanced history.** Most clients arrive in the last eight months, so every result rests on recent months and on the clients present then. The development pairs cover February to May 2026 only; seasonal effects cannot be separated from these four months.
- **Position is a daily average across queries.** A page's position each day already averages all the searches it appeared for, so expected clicks remain an approximation, though no longer one that overlaps the target month.
- **Optimizations are only partly visible.** Only each page's latest optimization is recorded, and all recorded dates fall from April 2026 onwards. Earlier optimizations cannot be seen, so some pages in the February and March pairs may have been edited without a trace.
- **The content table is a July 2026 snapshot.** Only fields fixed before a page is written (keyword research fields and content type) are used as features. Content type is assumed not to change.
- **GA4 is thin and not random.** Real engagement numbers exist only for a minority of page-months, clustered in the clients that connected GA4, so engagement stays a reason code and never ranks pages.
- **The negative binomial spread factor is one number for all pages.** It is estimated from pages with a stable position and applied to every page; a page-type-specific spread may fit better.
- **The sealed pair has not been checked.** Every number above comes from the development pairs. May → June will be read once, at the end.


## Self-check

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
